In [1]:
!pip install -q -U unsloth
!pip install transformers==5.5.0
!pip install -q -U accelerate bitsandbytes  pymupdf qwen-vl-utils[decord] regex pef

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 146.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import unsloth
from unsloth import FastVisionModel
import torchvision
import torchaudio
import torch
import pymupdf
import os
import json
from peft import PeftModel
import regex as r
from pathlib import Path
from qwen_vl_utils import process_vision_info
from dotenv import dotenv_values, load_dotenv

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
load_dotenv(Path.cwd().parent / ".env")

True

In [ ]:
model, processor = FastVisionModel.from_pretrained(os.getenv("BASE_OCR_MODEL"),
                                                   load_in_4bit=True,
                                                   device_map="auto",
                                                   )

FastVisionModel.for_inference(model)

model = PeftModel.from_pretrained(model, os.getenv("LORA_OCR_URL"))
model.eval()

In [ ]:
def pdf_to_images(pdf_path: str, out_dir: str, dpi: int = 200):
  """
  Parameters:
  - pdf_path: Path to the input PDF file.
  - out_dir: Directory where the output images will be saved.
  - dpi: Dots per inch for the output images.
  """

  os.makedirs(out_dir, exist_ok=True)
  doc = pymupdf.open(pdf_path)
  zoom = dpi / 72
  mat = pymupdf.Matrix(zoom, zoom)

  image_paths = []
  for i, page in enumerate(doc):
    pix = page.get_pixmap(matrix=mat)
    img_path = os.path.join(out_dir, f"page_{i+1:03d}.png")
    pix.save(img_path)
    image_paths.append(img_path)
    print(f"Rendered {img_path}")


  doc.close()
  return image_paths


def run_ocr(image_paths: list[str], model_id: str = os.getenv("BASE_OCR_MODEL")):
  """
  Parameters:
  - image_paths: List of paths to the images to be processed.
  - model_id: model identifier for the FastVisionModel.
  Returns:
  - results: A dictionary where keys are image paths and values are the OCR results in JSON
  """

  results = {}
  for img_path in image_paths:
      messages = [
      {
          "role": "user",
          "content": [
              {"type": "image", "image": f"{img_path}"},
              {"type": "text", "text":
                """Look at this image and determine if it contains an actual table — that is, structured content with visible rows
                  and columns, gridlines or clear column alignment, and multiple data fields per row. There may be zero, one, or
                  multiple seperate tables in this page, so don't stop after finding just one. Tables are usually seperated by
                  headers, whitespace, or a change in column structure. Extract them as seperate tables.

                  OUTPUT - the output should always be the "table_found" field followed by a  list of tables, even if there is just one.

                  Do NOT treat the following as tables:
                  - Body text or prose paragraphs, even if organized under headings
                  - Single-column lists of headings followed by descriptive text
                  - Definition-style content (a term followed by an explanatory paragraph)

                  If NO real table is present, respond with exactly:
                  {"table_found": false}

                  If a real table IS present, extract it with this format:
                  {
                    "table_found": true,
                    "tables" : [
                    {
                    "table_name": "<inferred from caption/context>",
                    "headers" : [header1, header2, ...],
                    "rows": [
                      ["row1val1", "row1val2", ...]
                        ]
                      }
                    ]
                  }


                  Rules:
                  - Output valid JSON only, no explanation before or after
                  - Preserve merged cells by repeating the value across merged rows/columns
                  - If a cell is empty or unreadable, use null
                  - Keep numbers as strings if they include currency symbols, %, or commas
                  - Always extract exactly what appears in the top row, cell by cell, even if it's just a date or number (i.e. Q3 2025 or 2026)
                  - In addition to the last rule, if a row's first cell is a date, "e.g. "At 31 December 2022", and has a number (e.g. 25,000 2300) in the same row, it is ALWAYS a data row and is to NEVER be a header — this is true even if it's position is the same as the header's
                  - A header row's cells are category or period labels only — they never carry a specific numeric value
                  - If there is no distinct header row at all, set column_headers to null
                  """}
              ]
          }
      ]

      text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
      image_inputs, video_inputs = process_vision_info(messages)
      inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

      out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
      trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
      results[img_path] = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
      print(f"OCR done: {img_path}")

  return results
